<a href="https://colab.research.google.com/github/04231073/Smart_City_MultiZone_Load_Forecasting/blob/main/model_A_Baseline_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

# Set device (GPU/CPU) & random seed agar hasil stabil
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"Menggunakan device: {device}")

Menggunakan device: cpu


In [ ]:
# 1. Load Dataset
df = pd.read_csv('Tetuan City power consumption.csv')
df.columns = df.columns.str.strip() # Hapus spasi ganda pada nama kolom

# 2. Feature Engineering DateTime
df['DateTime'] = pd.to_datetime(df['DateTime'])
df['Hour'] = df['DateTime'].dt.hour
df['DayOfWeek'] = df['DateTime'].dt.dayofweek
df['Month'] = df['DateTime'].dt.month

# Cyclical Encoding untuk Fitur Waktu
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24.0)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24.0)

print("Fitur DateTime berhasil diolah. 5 data pertama:")
df[['DateTime', 'Hour_sin', 'Hour_cos', 'DayOfWeek', 'Month']].head()

Fitur DateTime berhasil diolah. 5 data pertama:


,DateTime,Hour_sin,Hour_cos,DayOfWeek,Month
0,2017-01-01 00:00:00,0.0,1.0,6,1
1,2017-01-01 00:10:00,0.0,1.0,6,1
2,2017-01-01 00:20:00,0.0,1.0,6,1
3,2017-01-01 00:30:00,0.0,1.0,6,1
4,2017-01-01 00:40:00,0.0,1.0,6,1


In [ ]:
# 1. Tentukan Kolom Fitur dan Target
feature_cols = [
    'Temperature', 'Humidity', 'Wind Speed',
    'general diffuse flows', 'diffuse flows',
    'Hour_sin', 'Hour_cos', 'DayOfWeek', 'Month'
]
target_cols = [
    'Zone 1 Power Consumption',
    'Zone 2  Power Consumption',
    'Zone 3  Power Consumption'
]

X_raw = df[feature_cols].values
y_raw = df[target_cols].values

# 2. Scaling Data ke Rentang 0 - 1
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw)

# 3. Fungsi Sliding Window / Sequence Time-Series
def create_sequences(X, y, seq_length=144): # 144 langkah = 24 jam data
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i : i + seq_length])
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

SEQ_LEN = 144
X_seq, y_seq = create_sequences(X_scaled, y_scaled, seq_length=SEQ_LEN)
print(f"Bentuk data sequence X: {X_seq.shape}, y: {y_seq.shape}")

Bentuk data sequence X: (52272, 144, 9), y: (52272, 3)


In [ ]:
# Split 80% Train, 20% Test (tanpa shuffle karena time-series)
train_size = int(len(X_seq) * 0.8)

X_train = torch.tensor(X_seq[:train_size], dtype=torch.float32)
y_train = torch.tensor(y_seq[:train_size], dtype=torch.float32)
X_test = torch.tensor(X_seq[train_size:], dtype=torch.float32)
y_test = torch.tensor(y_seq[train_size:], dtype=torch.float32)

# Buat PyTorch DataLoader
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)

print(f"Jumlah sampel Train: {len(X_train)} | Jumlah sampel Test: {len(X_test)}")

Jumlah sampel Train: 41817 | Jumlah sampel Test: 10455


In [ ]:
class BaselineRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(BaselineRNN, self).__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Ambil output dari time-step terakhir
        return out

# Inisialisasi hyperparameter dasar Model A
input_dim = len(feature_cols)
hidden_dim = 64
output_dim = 3 # Predict Zone 1, Zone 2, Zone 3

model_A = BaselineRNN(input_dim, hidden_dim, output_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_A.parameters(), lr=0.001)

print(model_A)

BaselineRNN(
  (rnn): RNN(9, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=3, bias=True)
)


In [ ]:
epochs = 10
print("--- Memulai Pelatihan Model A (Baseline RNN) ---")

for epoch in range(epochs):
    # Mode Pelatihan
    model_A.train()
    train_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)

        optimizer.zero_grad()
        preds = model_A(X_b)
        loss = criterion(preds, y_b)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Mode Evaluasi
    model_A.eval()
    test_loss = 0
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            preds = model_A(X_b)
            test_loss += criterion(preds, y_b).item()

    avg_train_loss = train_loss / len(train_loader)
    avg_test_loss = test_loss / len(test_loader)
    print(f"Epoch {epoch+1:02d}/{epochs} | Train MSE: {avg_train_loss:.5f} | Test MSE: {avg_test_loss:.5f}")

--- Memulai Pelatihan Model A (Baseline RNN) ---
Epoch 01/10 | Train MSE: 0.01092 | Test MSE: 0.01106
Epoch 02/10 | Train MSE: 0.00507 | Test MSE: 0.01205
Epoch 03/10 | Train MSE: 0.00450 | Test MSE: 0.01196
Epoch 04/10 | Train MSE: 0.00417 | Test MSE: 0.01222
Epoch 05/10 | Train MSE: 0.00392 | Test MSE: 0.01229
Epoch 06/10 | Train MSE: 0.00373 | Test MSE: 0.01234
Epoch 07/10 | Train MSE: 0.00351 | Test MSE: 0.01216
Epoch 08/10 | Train MSE: 0.00330 | Test MSE: 0.01205
Epoch 09/10 | Train MSE: 0.00314 | Test MSE: 0.01210
Epoch 10/10 | Train MSE: 0.00303 | Test MSE: 0.01177
